# Uber pickups — KMeans, DBSCAN et zones chaudes

**Deuxième notebook dans l’ordre de lecture**, après `Uber_preparation_nico.ipynb`, depuis le dossier
`UBER_PICKUPS`, avec l'environnement `certif`. Bibliothèques : pandas, NumPy, GeoPandas, scikit-learn et Plotly.

Ce notebook est autonome : il recharge les fichiers bruts et reproduit la préparation du premier.
Les tableaux et les cartes sont conservés dans les sorties de cellules. Aucun export n’est créé.

Je garde une progression explicite : lire les données, tester les paramètres sur un jour,
entraîner les algorithmes par jour de semaine, comparer les résultats et produire les cartes.
Il n'y a ni pipeline ni fonction personnalisée.

Le projet sera terminé si les résultats permettent de :

- visualiser les zones chaudes de New York sur des cartes ;
- décrire ces zones pour chaque jour de semaine ;
- comparer KMeans et DBSCAN sur les mêmes positions ;
- consulter les cartes interactives et les tableaux directement dans les sorties du notebook.

Chaque cellule de rendu est précédée de **LIVRABLE** et porte aussi le tag `livrable`.

## 1. Importer les bibliothèques et lire les données

In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import plotly.express as px
import plotly.io as pio
from IPython.display import display
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score

pio.renderers.default = "plotly_mimetype"

In [2]:
files = [
    "data/uber-trip-data/uber-raw-data-apr14.csv",
    "data/uber-trip-data/uber-raw-data-may14.csv",
    "data/uber-trip-data/uber-raw-data-jun14.csv",
    "data/uber-trip-data/uber-raw-data-jul14.csv",
    "data/uber-trip-data/uber-raw-data-aug14.csv",
    "data/uber-trip-data/uber-raw-data-sep14.csv",
]

raw_2014 = pd.concat([pd.read_csv(file) for file in files], ignore_index=True)
raw_2014.columns = raw_2014.columns.str.lower()
raw_2014["date/time"] = pd.to_datetime(raw_2014["date/time"], format="%m/%d/%Y %H:%M:%S")

print("Raw rows / columns:", raw_2014.shape)

Raw rows / columns: (4534327, 4)


### Retrouver les prises en charge de New York

Je reprends la jointure du premier notebook avec les zones de taxi des boroughs de New York.
Je conserve les coordonnées GPS d'origine, puis j'échantillonne séparément chaque jour de semaine.

In [3]:
zones_official = gpd.read_file("data/taxi_zones/taxi_zones/taxi_zones.shp")
zones_official = zones_official[["LocationID", "zone", "borough", "geometry"]]
zones_official = zones_official.to_crs("EPSG:4326")

boroughs = ["Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island"]
zones_official = zones_official[zones_official["borough"].isin(boroughs)].copy()

loc_2014 = gpd.GeoDataFrame(
    raw_2014,
    geometry=gpd.points_from_xy(raw_2014["lon"], raw_2014["lat"]),
    crs="EPSG:4326",
)

dataset_2014 = gpd.sjoin(loc_2014, zones_official, how="inner", predicate="within")
dataset_2014 = dataset_2014.drop(columns=["index_right"]).reset_index(drop=True)

weekday_names = ["Lundi", "Mardi", "Mercredi", "Jeudi", "Vendredi", "Samedi", "Dimanche"]
dataset_2014["weekday_number"] = dataset_2014["date/time"].dt.dayofweek
dataset_2014["weekday"] = dataset_2014["weekday_number"].map(dict(enumerate(weekday_names)))
dataset_2014["hour"] = dataset_2014["date/time"].dt.hour

sample = dataset_2014.groupby("weekday_number", group_keys=False).sample(n=5000, random_state=42)
sample = sample.copy()

print("NYC pickups:", len(dataset_2014))
print("Sample rows:", len(sample))

NYC pickups: 4411894
Sample rows: 35000


### Exprimer les distances en kilomètres

La projection UTM 18N permet d'utiliser la même distance euclidienne pour les deux algorithmes.
Je garde les positions GPS d'origine pour les cartes.

In [4]:
sample_projected = sample.to_crs("EPSG:32618")
sample["x_km"] = sample_projected.geometry.x / 1000
sample["y_km"] = sample_projected.geometry.y / 1000
sample = pd.DataFrame(sample.drop(columns=["geometry"]))

numerical_features = ["x_km", "y_km"]

print("Missing coordinates:", sample[numerical_features].isna().sum().sum())
print("Distinct GPS positions:", len(sample[["lat", "lon"]].drop_duplicates()))
display(sample.head())
display(sample.groupby("weekday_number").size().rename("sample_pickups"))

Missing coordinates: 0
Distinct GPS positions: 29469


,date/time,lat,lon,base,LocationID,zone,borough,weekday_number,weekday,hour,x_km,y_km
1397427,2014-06-23 07:52:00,40.7226,-73.9982,B02598,144,Little Italy/NoLiTa,Manhattan,0,Lundi,7,584.605809,4508.446348
236975,2014-04-07 15:10:00,40.7623,-73.9680,B02617,237,Upper East Side South,Manhattan,0,Lundi,15,587.104542,4512.882871
3617892,2014-09-22 13:24:00,40.7591,-73.9774,B02598,161,Midtown Center,Manhattan,0,Lundi,13,586.315276,4512.518357
993518,2014-05-05 14:48:00,40.7245,-73.9975,B02682,144,Little Italy/NoLiTa,Manhattan,0,Lundi,14,584.662520,4508.657936
4400755,2014-09-29 13:22:00,40.8225,-73.9488,B02764,116,Hamilton Heights,Manhattan,0,Lundi,13,588.645009,4519.584805


weekday_number
0    5000
1    5000
2    5000
3    5000
4    5000
5    5000
6    5000
Name: sample_pickups, dtype: int64

Les coordonnées `x_km` et `y_km` sont les positions GPS projetées en kilomètres dans le système UTM 18N.
Les heures et les jours servent à sélectionner les trajets ; ils ne sont pas ajoutés aux coordonnées
dans le calcul de distance. Je cherche ainsi des regroupements géographiques pour un jour donné.

## 2. Tester KMeans sur un jour de semaine

Je commence par le lundi, toutes heures confondues. KMeans impose un nombre de groupes `k`
et affecte chaque point à l'un d'eux. Je reprends tes courbes d'inertie et de silhouette.
La silhouette est calculée sur un sous-échantillon reproductible pour éviter un calcul trop long.

In [5]:
selected_weekday = "Lundi"
one_day = sample[sample["weekday"] == selected_weekday].copy()
X = one_day[numerical_features]

inertia = []
silhouette_scores = []
K = range(2, 11)

for k in K:
    kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
    kmeans.fit(X)
    inertia.append(kmeans.inertia_)
    score = silhouette_score(X, kmeans.labels_, sample_size=1000, random_state=42)
    silhouette_scores.append(score)

kmeans_search = pd.DataFrame({
    "k": list(K),
    "inertia": inertia,
    "silhouette": silhouette_scores,
})
display(kmeans_search.round(3))

,k,inertia,silhouette
0,2,98846.708,0.718
1,3,59237.991,0.437
2,4,40845.106,0.471
3,5,28595.065,0.475
4,6,22910.206,0.445
5,7,19481.825,0.477
6,8,16084.468,0.461
7,9,14057.963,0.427
8,10,12118.677,0.434


In [6]:
fig_elbow = px.line(
    kmeans_search,
    x="k",
    y="inertia",
    markers=True,
    title="KMeans : évolution de l'inertie le lundi",
    labels={"k": "Nombre de clusters", "inertia": "Inertie"},
)
fig_elbow.show()

fig_silhouette = px.line(
    kmeans_search,
    x="k",
    y="silhouette",
    markers=True,
    title="KMeans : silhouette le lundi",
    labels={"k": "Nombre de clusters", "silhouette": "Score de silhouette"},
)
fig_silhouette.show()

Je retiens **`k=5`** comme compromis de lecture : l'inertie diminue nettement jusqu'à ce niveau,
puis les gains sont plus modestes. Le meilleur score de silhouette est obtenu avec deux groupes,
mais ce découpage est trop grossier pour décrire plusieurs lieux où se positionner.
Le choix de cinq groupes reste un choix d'analyse, pas un optimum démontré.

## 3. Tester DBSCAN sur les mêmes points

DBSCAN cherche des regroupements denses sans imposer leur nombre.
`eps` est un rayon de voisinage en kilomètres. `min_samples` est le nombre minimal
de points dans ce voisinage, point central compris, pour former un point cœur.
Le label **`-1` désigne le bruit** : ces points ne sont rattachés à aucun cluster.

In [7]:
dbscan_results = []

for eps in [0.10, 0.15, 0.20, 0.30, 0.50]:
    dbscan = DBSCAN(eps=eps, min_samples=20, metric="euclidean")
    labels = dbscan.fit_predict(X)

    cluster_count = len(set(labels) - {-1})
    noise_pct = (labels == -1).mean() * 100
    largest_cluster = pd.Series(labels[labels != -1]).value_counts().max()

    dbscan_results.append({
        "eps_km": eps,
        "min_samples": 20,
        "clusters": cluster_count,
        "noise_pct": noise_pct,
        "largest_cluster_pickups": largest_cluster,
    })

dbscan_search = pd.DataFrame(dbscan_results)
display(dbscan_search.round(2))

,eps_km,min_samples,clusters,noise_pct,largest_cluster_pickups
0,0.10,20,15,88.88,84
1,0.15,20,25,62.28,739
2,0.20,20,19,36.92,2180
3,0.30,20,6,20.54,3515
4,0.50,20,5,13.74,3735


Je retiens **`eps=0.15` km et `min_samples=20`** pour repérer des concentrations locales.
Le tableau montre le compromis : un rayon plus grand rattache davantage de points,
mais fusionne une grande partie de Manhattan dans un seul groupe.
Le réglage retenu laisse beaucoup de bruit ; ce taux sera affiché dans la comparaison.

Ces paramètres s'appliquent à notre échantillon de 5 000 trajets par jour de semaine.
Ils devraient être réévalués si la taille de l'échantillon ou la période change.

## 4. Entraîner les deux algorithmes pour chaque jour de semaine

Je garde les mêmes paramètres pour tous les jours afin de comparer des analyses construites
de la même façon. Chaque jour possède ses propres clusters : **un numéro de cluster n'identifie
pas un quartier stable d'un jour à l'autre**.

In [8]:
n_clusters = 5
eps_km = 0.15
min_samples = 20

clustered_days = []

for weekday in weekday_names:
    day_sample = sample[sample["weekday"] == weekday].copy()
    X_day = day_sample[numerical_features]

    kmeans = KMeans(n_clusters=n_clusters, n_init=10, random_state=42)
    day_sample["kmeans_cluster"] = kmeans.fit_predict(X_day)

    dbscan = DBSCAN(eps=eps_km, min_samples=min_samples, metric="euclidean")
    day_sample["dbscan_cluster"] = dbscan.fit_predict(X_day)

    clustered_days.append(day_sample)

sample_clusters = pd.concat(clustered_days, ignore_index=True)
display(sample_clusters.head())

,date/time,lat,lon,base,LocationID,zone,borough,weekday_number,weekday,hour,x_km,y_km,kmeans_cluster,dbscan_cluster
0,2014-06-23 07:52:00,40.7226,-73.9982,B02598,144,Little Italy/NoLiTa,Manhattan,0,Lundi,7,584.605809,4508.446348,4,0
1,2014-04-07 15:10:00,40.7623,-73.9680,B02617,237,Upper East Side South,Manhattan,0,Lundi,15,587.104542,4512.882871,1,-1
2,2014-09-22 13:24:00,40.7591,-73.9774,B02598,161,Midtown Center,Manhattan,0,Lundi,13,586.315276,4512.518357,1,1
3,2014-05-05 14:48:00,40.7245,-73.9975,B02682,144,Little Italy/NoLiTa,Manhattan,0,Lundi,14,584.662520,4508.657936,4,0
4,2014-09-29 13:22:00,40.8225,-73.9488,B02764,116,Hamilton Heights,Manhattan,0,Lundi,13,588.645009,4519.584805,1,-1


## 5. Comparer les résultats

Je compare le nombre de clusters, la couverture et le bruit.
Pour la silhouette, j'utilise **les mêmes points pour les deux algorithmes** : ceux que DBSCAN
ne classe pas comme bruit. Un sous-échantillon commun est tiré avec la même graine.

Ainsi, DBSCAN ne bénéficie pas d'un score calculé sur un ensemble de points différent de KMeans.
La silhouette favorise néanmoins les formes compactes et ne suffit pas à choisir une méthode
de détection des zones chaudes. Un score non calculable reste vide.

In [9]:
comparison_rows = []

for weekday in weekday_names:
    day_sample = sample_clusters[sample_clusters["weekday"] == weekday].copy()
    common_points = day_sample[day_sample["dbscan_cluster"] != -1].copy()
    score_points = common_points.sample(n=min(1000, len(common_points)), random_state=42)

    for algorithm, cluster_column in [
        ("KMeans", "kmeans_cluster"),
        ("DBSCAN", "dbscan_cluster"),
    ]:
        labels = day_sample[cluster_column]
        score_labels = score_points[cluster_column]
        score = np.nan

        if 2 <= score_labels.nunique() < len(score_points):
            score = silhouette_score(score_points[numerical_features], score_labels)

        comparison_rows.append({
            "weekday": weekday,
            "algorithm": algorithm,
            "sample_pickups": len(day_sample),
            "clusters": len(set(labels) - {-1}),
            "noise_pct": (labels == -1).mean() * 100,
            "coverage_pct": (labels != -1).mean() * 100,
            "common_score_pickups": len(score_points),
            "silhouette_common_points": score,
        })

comparison = pd.DataFrame(comparison_rows)

### LIVRABLE — Comparaison KMeans / DBSCAN

**La cellule suivante produit** le tableau comparatif pour tous les jours de semaine
et le graphique de couverture de chaque méthode. Les résultats restent dans les sorties de la cellule.

In [10]:
display(comparison.round(3))

fig_comparison = px.bar(
    comparison,
    x="weekday",
    y="coverage_pct",
    color="algorithm",
    barmode="group",
    category_orders={"weekday": weekday_names},
    labels={"weekday": "Jour", "coverage_pct": "Points rattachés à un cluster (%)", "algorithm": "Algorithme"},
    title="Couverture des deux algorithmes par jour de semaine",
)
fig_comparison.update_yaxes(range=[0, 105])
fig_comparison.show()

,weekday,algorithm,sample_pickups,clusters,noise_pct,coverage_pct,common_score_pickups,silhouette_common_points
0,Lundi,KMeans,5000,5,0.00,100.00,1000,0.631
1,Lundi,DBSCAN,5000,25,62.28,37.72,1000,0.410
2,Mardi,KMeans,5000,5,0.00,100.00,1000,0.562
3,Mardi,DBSCAN,5000,30,54.84,45.16,1000,0.281
4,Mercredi,KMeans,5000,5,0.00,100.00,1000,0.565
5,Mercredi,DBSCAN,5000,19,58.30,41.70,1000,0.292
6,Jeudi,KMeans,5000,5,0.00,100.00,1000,0.605
7,Jeudi,DBSCAN,5000,25,52.16,47.84,1000,0.200
8,Vendredi,KMeans,5000,5,0.00,100.00,1000,0.593
9,Vendredi,DBSCAN,5000,22,64.38,35.62,1000,0.331


## 6. Décrire et classer les zones chaudes

Pour chaque cluster, je compte les trajets de l'échantillon et je calcule leur part parmi
tous les points du jour, bruit compris. Je trie ensuite les clusters par fréquentation.
Ce classement décrit une concentration historique ; il ne mesure pas les demandes non satisfaites.

Pour placer un repère, je prends **un vrai point du cluster, le plus proche de sa moyenne**.
Cela évite de présenter une moyenne GPS située hors des lieux observés.
La zone et le borough indiqués sont ceux de ce point ; `main_zone` désigne la zone de taxi
la plus fréquente du cluster. Un cluster peut couvrir plusieurs zones de taxi.

In [11]:
hot_zone_rows = []

for weekday in weekday_names:
    day_sample = sample_clusters[sample_clusters["weekday"] == weekday].copy()

    for algorithm, cluster_column in [
        ("KMeans", "kmeans_cluster"),
        ("DBSCAN", "dbscan_cluster"),
    ]:
        clustered_points = day_sample[day_sample[cluster_column] != -1]

        for cluster_id, cluster_points in clustered_points.groupby(cluster_column):
            center_x = cluster_points["x_km"].mean()
            center_y = cluster_points["y_km"].mean()
            squared_distance = (
                (cluster_points["x_km"] - center_x) ** 2
                + (cluster_points["y_km"] - center_y) ** 2
            )
            representative = cluster_points.loc[squared_distance.idxmin()]

            hot_zone_rows.append({
                "weekday": weekday,
                "algorithm": algorithm,
                "cluster": int(cluster_id),
                "sample_pickups": len(cluster_points),
                "share_pct": len(cluster_points) / len(day_sample) * 100,
                "main_zone": cluster_points["zone"].value_counts().index[0],
                "marker_zone": representative["zone"],
                "borough": representative["borough"],
                "lat": representative["lat"],
                "lon": representative["lon"],
            })

hot_zones = pd.DataFrame(hot_zone_rows)
hot_zones["weekday"] = pd.Categorical(hot_zones["weekday"], categories=weekday_names, ordered=True)
hot_zones = hot_zones.sort_values(
    ["weekday", "algorithm", "sample_pickups", "cluster"],
    ascending=[True, True, False, True],
)
hot_zones["rank"] = hot_zones.groupby(["weekday", "algorithm"], observed=True).cumcount() + 1
top_hot_zones = hot_zones[hot_zones["rank"] <= 3].copy()

### LIVRABLE — Zones chaudes par jour de semaine

**La cellule suivante produit** les trois clusters les plus fréquentés de chaque méthode
pour chaque jour, avec leurs quartiers et coordonnées. Le tableau est affiché dans le notebook.

In [12]:
display_columns = [
    "weekday", "algorithm", "rank", "main_zone", "marker_zone",
    "borough", "sample_pickups", "share_pct", "lat", "lon",
]
display(top_hot_zones[display_columns].round({"share_pct": 2, "lat": 4, "lon": 4}))

,weekday,algorithm,rank,main_zone,marker_zone,borough,sample_pickups,share_pct,lat,lon
6,Lundi,DBSCAN,1,Midtown Center,Midtown Center,Manhattan,739,14.78,40.7584,-73.9775
5,Lundi,DBSCAN,2,SoHo,SoHo,Manhattan,175,3.50,40.7241,-73.9981
8,Lundi,DBSCAN,3,Meatpacking/West Village West,Meatpacking/West Village West,Manhattan,102,2.04,40.7403,-74.0058
1,Lundi,KMeans,1,Midtown Center,Midtown North,Manhattan,2067,41.34,40.7650,-73.9747
4,Lundi,KMeans,2,Union Sq,Greenwich Village North,Manhattan,1948,38.96,40.7295,-73.9945
0,Lundi,KMeans,3,Park Slope,Prospect Heights,Brooklyn,475,9.50,40.6773,-73.9672
38,Mardi,DBSCAN,1,Midtown Center,Midtown Center,Manhattan,826,16.52,40.7582,-73.9774
37,Mardi,DBSCAN,2,Union Sq,Union Sq,Manhattan,279,5.58,40.7418,-73.9895
39,Mardi,DBSCAN,3,SoHo,SoHo,Manhattan,266,5.32,40.7233,-73.9997
31,Mardi,KMeans,1,Midtown Center,Midtown North,Manhattan,2134,42.68,40.7638,-73.9748


## 7. Cartographier les clusters

Ces cartes montrent les **points réellement utilisés pour l'entraînement**.
Le curseur permet de changer de jour de semaine. Sur la carte DBSCAN, le bruit reste visible
dans la catégorie `Noise`, mais il n'entre pas dans les recommandations.
Les fonds cartographiques nécessitent une connexion Internet.

Pour l'animation DBSCAN, je réserve les mêmes catégories dans chaque journée.
Les catégories absentes gardent des coordonnées vides uniquement pour l'affichage :
cela évite de mélanger les traces lorsque le nombre de clusters change.

In [13]:
sample_clusters["kmeans_label"] = "Cluster " + sample_clusters["kmeans_cluster"].astype(str)
sample_clusters["dbscan_label"] = "Cluster " + sample_clusters["dbscan_cluster"].astype(str)
sample_clusters.loc[sample_clusters["dbscan_cluster"] == -1, "dbscan_label"] = "Noise"

map_center = {"lat": sample["lat"].median(), "lon": sample["lon"].median()}

dbscan_cluster_ids = sorted(sample_clusters.loc[sample_clusters["dbscan_cluster"] != -1, "dbscan_cluster"].unique())
dbscan_label_order = ["Noise"] + [f"Cluster {cluster_id}" for cluster_id in dbscan_cluster_ids]

map_slots = []
for weekday in weekday_names:
    for cluster_label in dbscan_label_order:
        map_slots.append({"weekday": weekday, "dbscan_label": cluster_label})

dbscan_map_data = pd.DataFrame(map_slots).merge(
    sample_clusters,
    on=["weekday", "dbscan_label"],
    how="left",
    validate="one_to_many",
)

### LIVRABLE — Carte des clusters KMeans

**La cellule suivante produit** la carte interactive KMeans par jour de semaine.
Les couleurs distinguent les clusters du jour affiché.

In [14]:
fig_kmeans = px.scatter_map(
    sample_clusters,
    lat="lat",
    lon="lon",
    color="kmeans_label",
    animation_frame="weekday",
    category_orders={"weekday": weekday_names},
    hover_name="zone",
    hover_data={"borough": True, "hour": True, "lat": ":.4f", "lon": ":.4f"},
    labels={"weekday": "Jour", "kmeans_label": "Cluster", "hour": "Heure", "borough": "Borough"},
    opacity=0.55,
    map_style="carto-positron",
    center=map_center,
    zoom=10,
    height=650,
    title="KMeans — prises en charge regroupées par jour de semaine",
)
fig_kmeans.show()

### LIVRABLE — Carte des clusters DBSCAN

**La cellule suivante produit** la carte interactive DBSCAN par jour de semaine.
La légende permet de masquer le bruit ou de sélectionner un cluster.

In [15]:
fig_dbscan = px.scatter_map(
    dbscan_map_data,
    lat="lat",
    lon="lon",
    color="dbscan_label",
    animation_frame="weekday",
    category_orders={"weekday": weekday_names, "dbscan_label": dbscan_label_order},
    hover_name="zone",
    hover_data={"borough": True, "hour": True, "lat": ":.4f", "lon": ":.4f"},
    labels={"weekday": "Jour", "dbscan_label": "Cluster", "hour": "Heure", "borough": "Borough"},
    opacity=0.55,
    map_style="carto-positron",
    center=map_center,
    zoom=10,
    height=650,
    title="DBSCAN — concentrations locales et points non rattachés",
)
fig_dbscan.show()

## 8. Afficher les repères des zones les plus fréquentées

Les numéros de rang représentent la fréquentation relative du jour : le rang 1 est le cluster
avec le plus de trajets dans l'échantillon. Deux cartes permettront de comparer les repères
de KMeans et DBSCAN.

In [16]:
top_hot_zones["zone_rank"] = "Rang " + top_hot_zones["rank"].astype(str)
hot_zone_maps = {}

for algorithm in ["KMeans", "DBSCAN"]:
    map_data = top_hot_zones[top_hot_zones["algorithm"] == algorithm].copy()

    fig_hot_zones = px.scatter_map(
        map_data,
        lat="lat",
        lon="lon",
        size="sample_pickups",
        color="zone_rank",
        animation_frame="weekday",
        animation_group="zone_rank",
        category_orders={"weekday": weekday_names, "zone_rank": ["Rang 1", "Rang 2", "Rang 3"]},
        hover_name="main_zone",
        hover_data={
            "marker_zone": True,
            "borough": True,
            "sample_pickups": True,
            "share_pct": ":.2f",
            "lat": ":.4f",
            "lon": ":.4f",
        },
        labels={
            "weekday": "Jour",
            "zone_rank": "Classement du jour",
            "marker_zone": "Quartier du repère",
            "borough": "Borough du repère",
            "sample_pickups": "Trajets de l'échantillon",
            "share_pct": "Part du jour (%)",
        },
        size_max=35,
        map_style="carto-positron",
        center=map_center,
        zoom=10,
        height=600,
        title=f"{algorithm} — trois zones chaudes par jour de semaine",
    )
    hot_zone_maps[algorithm] = fig_hot_zones

### LIVRABLE — Cartes des zones chaudes recommandées

**La cellule suivante produit** les cartes des trois premiers repères de chaque jour.
Survoler un repère affiche le quartier et sa part des trajets échantillonnés.

In [17]:
hot_zone_maps["KMeans"].show()
hot_zone_maps["DBSCAN"].show()

## 9. Décrire les zones pour chaque jour

Je retiens DBSCAN pour présenter des concentrations locales de prises en charge.
KMeans fournit une vue plus large qui couvre tous les points.
Le classement ci-dessous restitue les résultats calculés, jour par jour.

### LIVRABLE — Description quotidienne des zones chaudes

**La cellule suivante produit** une description des trois premières zones DBSCAN de chaque jour,
en indiquant la part des trajets regroupés dans ces trois clusters.

In [18]:
for weekday in weekday_names:
    day_hot_zones = top_hot_zones[
        (top_hot_zones["weekday"] == weekday)
        & (top_hot_zones["algorithm"] == "DBSCAN")
    ]
    zone_descriptions = []

    for row in day_hot_zones.itertuples():
        zone_descriptions.append(f"{row.main_zone} ({row.share_pct:.1f} %)")

    total_share = day_hot_zones["share_pct"].sum()
    print(f"{weekday} : " + "; ".join(zone_descriptions))
    print(f"Part cumulée des trois clusters : {total_share:.1f} % des trajets du jour dans l'échantillon.")
    print()

Lundi : Midtown Center (14.8 %); SoHo (3.5 %); Meatpacking/West Village West (2.0 %)
Part cumulée des trois clusters : 20.3 % des trajets du jour dans l'échantillon.

Mardi : Midtown Center (16.5 %); Union Sq (5.6 %); SoHo (5.3 %)
Part cumulée des trois clusters : 27.4 % des trajets du jour dans l'échantillon.

Mercredi : Midtown Center (17.0 %); SoHo (5.0 %); Midtown South (3.5 %)
Part cumulée des trois clusters : 25.5 % des trajets du jour dans l'échantillon.

Jeudi : Midtown Center (17.4 %); SoHo (8.9 %); Union Sq (4.4 %)
Part cumulée des trois clusters : 30.7 % des trajets du jour dans l'échantillon.

Vendredi : SoHo (9.2 %); Midtown Center (7.7 %); Union Sq (2.6 %)
Part cumulée des trois clusters : 19.5 % des trajets du jour dans l'échantillon.

Samedi : Meatpacking/West Village West (3.0 %); SoHo (2.9 %); West Village (2.0 %)
Part cumulée des trois clusters : 7.9 % des trajets du jour dans l'échantillon.

Dimanche : Meatpacking/West Village West (3.5 %); Lower East Side (2.5 %); 

## Conclusion et limites

**KMeans** est facile à lire et rattache tous les trajets à un cluster, mais il impose leur nombre
et peut regrouper des zones étendues. **DBSCAN** fait ressortir des concentrations locales
et écarte des points isolés ; son résultat dépend fortement du rayon et du seuil de densité.
Son bruit élevé apparaît dans les résultats et ne signifie pas que ces lieux n'ont aucune demande.

Pour ce projet, je présente les repères DBSCAN comme des pistes de positionnement issues de
la fréquentation historique, en gardant KMeans pour la comparaison. Les résultats sont détaillés
pour chaque jour de semaine ; ils ne décrivent pas une heure précise. La carte de chaleur temporelle
complète la lecture des horaires, sans constituer un clustering par heure.

Les courses réalisées ne permettent pas à elles seules de mesurer l'attente, les demandes annulées
ou l'offre de chauffeurs. Les paramètres ont été explorés sur le lundi, puis appliqués aux autres jours ;
il ne s'agit pas d'une validation prédictive ou d'une optimisation exhaustive.

### Références techniques consultées

- [Clustering : KMeans et DBSCAN — scikit-learn](https://scikit-learn.org/stable/modules/clustering.html)
- [Paramètres de DBSCAN — scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.DBSCAN.html)
- [Score de silhouette — scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html)
- [Cartes de points — Plotly](https://plotly.com/python/map-scatter-plots/)